데이터 준비: 순수 CIFAR-10을 세 가지 버전으로

(a) 정상 이미지
(b) 흐릿하고 흑백인 이미지(초반 학습용)
(c) 깨끗한 테스트용. 색깔이 클래스랑 가짜로 엮인 것도 없고, align/conflict 구분도 없어. 그냥 평범한 CIFAR-10.

두 가지 모델을 각각 10개씩 학습:
- model_w: "발달적 학습법" 적용 — 랜덤 노이즈로 5 epoch 먼저 워밍업 → 처음 10 epoch은 흐릿하고 흑백인 이미지로 학습 → 그 이후 90 epoch은 정상 이미지로 학습. 중간(epoch 5)에 학습 속도(lr)를 한 번 낮춤.
- model_wo: 아무 특별한 처리 없이 처음부터 끝까지 정상 이미지로만 100 epoch 학습.
두 조건 다 test 정확도가 제일 높았던 순간의 모델 하나만 저장해둬(전체 epoch 다 저장하지 않음).

평가는 네 갈래로 나눠서 함:
- 깨끗한 테스트 정확도 비교: 그냥 평범한 CIFAR-10 테스트셋에서 두 모델이 얼마나 정확한지 비교 (기본적인 성능 손해가 없는지 확인).
- CIFAR-10-C(진짜 손상된 이미지) 평가: Hendrycks가 만든 노이즈/블러/날씨효과/압축손상 등으로 미리 망가뜨린 이미지들에 두 모델을 넣어서, 종류별로 얼마나 잘 버티는지 비교. 학습 때 쓴 블러랑 겹치는 종류(defocus_blur 등)는 따로 표시해서, "원래 훈련 때 본 거라 잘하는 건지" vs "안 본 손상에도 잘 버티는 건지"를 구분할 수 있게 해놨어.
- Calibration/OOD 탐지 벤치마크: random-noise warm-up(R)이 원래 노리는 효과(확신도 보정 + 낯선 데이터 탐지)가 G, L이랑 섞인 지금도 남아있는지, ECE랑 SVHN 기준 AUROC로 확인.
- 표상(feature) 분석: 마지막 층에서 뽑은 특징이 10개 클래스를 얼마나 잘 구분하는지(숫자로), 그리고 그걸 그림(t-SNE)으로도 확인.

흐릿하고 흑백인 상태로 학습을 시작한 모델이, 정상적으로만 학습한 모델보다 실제로 손상된 이미지를 더 잘 알아보는가?

In [ ]:
import sys
import os
import tarfile
import urllib.request
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import copy
import glob
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader

sys.path.append('..')

from src.data.degredation import get_transforms
from src.models.resnet import resnet18
from src.training.train import train, test

from custom.figure import mm, color
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = "cifar10"
dataset_dir = '../dataset'


class AttrlessDataset(torch.utils.data.Dataset):
    """Wraps a plain (image, label) dataset so it still yields a placeholder
    bias-attribute, matching the (image, label, attr) triple that
    train()/test() -- and the other helpers originally written for the
    bias-conflict CIFAR-10 dataset -- expect.

    Plain CIFAR-10 has no real bias attribute, so -1 is just a stand-in
    that keeps those functions runnable. It is never meant to be read as a
    meaningful signal -- nothing below should compute anything from it.
    """

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, label = self.base_dataset[index]
        return image, label, -1


train_dataset = AttrlessDataset(torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=False),
))
train_degradation_dataset = AttrlessDataset(torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=7, color=0, test=False),
))
test_dataset = AttrlessDataset(torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=True),
))

print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Train Degradation Dataset Size: {len(train_degradation_dataset)}")
print(f"Clean Test Dataset Size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
train_degradation_loader = DataLoader(train_degradation_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)


In [ ]:
dir = "cifar10"
num_net = 10

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", dir)
if not os.path.exists(figure_dir):
    os.makedirs(figure_dir)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)


In [ ]:
criterion = nn.CrossEntropyLoss()

epochs = 100

# LR suppression: applied ONCE, at lr_suppress_start -- not every epoch in a
# window. The previous version did `lr *= alpha` on every epoch between
# lr_suppress_start and lr_suppress_end (5 epochs), which compounds into
# lr ~= 1e-9 (0.0001 * 0.1**5) by the end of the window -- effectively
# freezing the network for the rest of training rather than just lowering
# its plasticity. Now it steps down once and stays there.
alpha = 0.01
lr_suppress_start = epochs // 20  # epoch 5 -- one-time step down happens here


In [ ]:
model_wo = [resnet18().to(device) for _ in range(num_net)]
model_w = [resnet18().to(device) for _ in range(num_net)]
optimizer_wo = [torch.optim.Adam(model_wo[net_idx].parameters(), lr=0.0001) for net_idx in range(num_net)]
optimizer_w = [torch.optim.Adam(model_w[net_idx].parameters(), lr=0.01) for net_idx in range(num_net)]
schedular_wo = [torch.optim.lr_scheduler.StepLR(optimizer_wo[net_idx], step_size=1000, gamma=0.1) for net_idx in range(num_net)] # epoch보다 크게 해서 적용 안되게 하기
schedular_w = [torch.optim.lr_scheduler.StepLR(optimizer_w[net_idx], step_size=1000, gamma=0.1) for net_idx in range(num_net)]

# Dropped the test_align_loss/test_align_acc / test_conflict_* fields:
# those tracked the bias-conflict split, which does not exist for plain
# CIFAR-10 (the earlier draft aliased them straight to test_loader, which
# just duplicated test_acc under a different name).
training_info = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "best_net": None}

training_info_wo = [copy.deepcopy(training_info) for _ in range(num_net)]
training_info_w = [copy.deepcopy(training_info) for _ in range(num_net)]


In [ ]:
# random-noise warm-up (model_w only -- independent pretraining, separate
# from the real-data epoch count)
from src.training.random_training import random_train

epochs_noise = 5
num_noise = 50000
input_shape = (3, 32, 32)   # CIFAR-10
output_size = 10

optimizer_noise = [torch.optim.SGD(model_w[net_idx].parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4) for net_idx in range(num_net)]

for net_idx in range(num_net):
    print(f"Random-noise warm-up Network {net_idx+1}/{num_net}")
    for epoch in range(epochs_noise):
        noise_loss, noise_acc = random_train(
            model_w[net_idx], optimizer_noise[net_idx], criterion,
            input_shape, num_noise, batch_size=128, output_size=output_size,
            device=device
        )
        print(f"  Warm-up Epoch {epoch+1}/{epochs_noise} - Loss: {noise_loss:.4f}, Acc: {noise_acc:.4f}")


In [ ]:
# training with degradation
for net_idx in range(num_net):
    print(f"Training Network {net_idx+1}/{num_net} with degradation")

    best_acc_w = 0.0

    for epoch in range(epochs + 1):
        if epoch == lr_suppress_start:
            for group in optimizer_w[net_idx].param_groups:
                group['lr'] *= alpha
            print(f"  [LR suppression] lr set to {optimizer_w[net_idx].param_groups[0]['lr']:.2e} at epoch {epoch}")

        if epoch == 0:
            train_loss, train_acc = test(model_w[net_idx], train_degradation_loader, criterion, device)
        elif epoch < epochs // 10:
            train_loss, train_acc = train(model_w[net_idx], train_degradation_loader, optimizer_w[net_idx], criterion, schedular_w[net_idx], device)
        else:
            train_loss, train_acc = train(model_w[net_idx], train_loader, optimizer_w[net_idx], criterion, schedular_w[net_idx], device)

        test_loss, test_acc = test(model_w[net_idx], test_loader, criterion, device)

        training_info_w[net_idx]["train_loss"].append(train_loss)
        training_info_w[net_idx]["train_acc"].append(train_acc)
        training_info_w[net_idx]["test_loss"].append(test_loss)
        training_info_w[net_idx]["test_acc"].append(test_acc)

        if test_acc > best_acc_w:
            best_acc_w = test_acc
            training_info_w[net_idx]["best_net"] = copy.deepcopy(model_w[net_idx].state_dict())
            torch.save(model_w[net_idx].state_dict(), os.path.join(save_dir, f"best_model_w_{net_idx}.pth"))

        print(f"Epoch {epoch+1}/{epochs} - w: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Clean Test Acc: {test_acc:.4f}")


In [ ]:
# training without degradation
for net_idx in range(num_net):
    print(f"Training Network {net_idx+1}/{num_net} without degradation")

    best_acc_wo = 0.0

    for epoch in range(epochs + 1):
        if epoch == 0:
            train_loss, train_acc = test(model_wo[net_idx], train_loader, criterion, device)
        else:
            train_loss, train_acc = train(model_wo[net_idx], train_loader, optimizer_wo[net_idx], criterion, schedular_wo[net_idx], device)

        test_loss, test_acc = test(model_wo[net_idx], test_loader, criterion, device)

        training_info_wo[net_idx]["train_loss"].append(train_loss)
        training_info_wo[net_idx]["train_acc"].append(train_acc)
        training_info_wo[net_idx]["test_loss"].append(test_loss)
        training_info_wo[net_idx]["test_acc"].append(test_acc)

        if test_acc > best_acc_wo:
            best_acc_wo = test_acc
            training_info_wo[net_idx]["best_net"] = copy.deepcopy(model_wo[net_idx].state_dict())
            torch.save(model_wo[net_idx].state_dict(), os.path.join(save_dir, f"best_model_wo_{net_idx}.pth"))

        print(f"Epoch {epoch+1}/{epochs} - wo: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Clean Test Acc: {test_acc:.4f}")


## Clean test accuracy comparison

In [ ]:
from scipy.stats import ranksums, ttest_ind

best_acc_w = [max(training_info_w[net_idx]["test_acc"]) for net_idx in range(num_net)]
best_acc_wo = [max(training_info_wo[net_idx]["test_acc"]) for net_idx in range(num_net)]

print(f"Best clean test accuracy w/o degradation: {np.mean(best_acc_wo)*100:.4f} (+/- {np.std(best_acc_wo)*100:.4f})")
print(f"Best clean test accuracy w/ degradation:  {np.mean(best_acc_w)*100:.4f} (+/- {np.std(best_acc_w)*100:.4f})")
print(ttest_ind(best_acc_wo, best_acc_w))
print(ranksums(best_acc_wo, best_acc_w))


## CIFAR-10-C corruption evaluation

Download CIFAR-10-C separately and extract it to `../dataset/CIFAR-10-C`
before running this cell (https://zenodo.org/records/2535967).


In [ ]:
cifar10c_root = os.path.join(dataset_dir, "CIFAR-10-C")
cifar10c_url = "https://zenodo.org/records/2535967/files/CIFAR-10-C.tar?download=1"
labels_path = os.path.join(cifar10c_root, "labels.npy")
tar_path = os.path.join(dataset_dir, "CIFAR-10-C.tar")

# FIX 1: the previous check was `if not os.path.isdir(cifar10c_root)`, which
# skips the download whenever the folder exists at all -- even if it's
# empty or only partially extracted (which is exactly what happened:
# FileNotFoundError on labels.npy even though the folder was already there).
# Checking for labels.npy itself is what actually tells us whether the data
# is usable.
#
# FIX 2: even after that, `if not os.path.exists(tar_path)` alone isn't
# enough -- if an earlier download got interrupted partway (very possible
# on a slow connection: the 170MB CIFAR-10 download earlier took ~17
# minutes, so this ~2.9GB file could take multiple hours), tar_path exists
# but is a truncated/corrupt file. Extracting a truncated tar can silently
# produce only some of the files, which is exactly how labels.npy could end
# up missing. So: if tar_path exists, verify it's actually a valid, complete
# archive before trusting it; delete and re-download if not.
if not os.path.exists(labels_path):
    need_download = True
    if os.path.exists(tar_path):
        try:
            with tarfile.open(tar_path) as tar:
                tar.getmembers()  # walks the whole archive; raises if truncated/corrupt
            need_download = False
        except (tarfile.TarError, EOFError):
            print("기존 CIFAR-10-C.tar이 손상된(다운로드가 중간에 끊긴) 것 같아서 다시 받을게.")
            os.remove(tar_path)

    if need_download:
        print("Downloading CIFAR-10-C (~2.9GB) -- 네트워크 속도에 따라 몇 시간 걸릴 수 있어, 창 닫지 말고 기다려줘.")
        urllib.request.urlretrieve(cifar10c_url, tar_path)

    print("Extracting CIFAR-10-C...")
    with tarfile.open(tar_path) as tar:
        tar.extractall(dataset_dir)
    print("Done.")

if not os.path.exists(labels_path):
    raise FileNotFoundError(
        f"Extraction did not produce {labels_path} -- check the archive's "
        f"internal folder name and adjust cifar10c_root if needed."
    )

labels_c = np.load(labels_path)
corruption_paths = sorted(
    path for path in glob.glob(os.path.join(cifar10c_root, "*.npy"))
    if not path.endswith("labels.npy")
)

cifar_mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
cifar_std = torch.tensor([0.2023, 0.1994, 0.2010], device=device).view(1, 3, 1, 1)

# The training-time degradation curriculum uses Gaussian blur, so doing
# well specifically on the blur-family corruptions below is a weaker claim
# (the model has literally seen that kind of distortion before) than doing
# well on noise/weather/digital corruptions it never saw. Kept as its own
# group so the two don't get averaged together and read as "generalizes."
blur_corruptions = {"defocus_blur", "glass_blur", "motion_blur", "zoom_blur"}

# These 4 are the "extra" corruptions in the public release, not part of
# the standard 15 used in most published mCE comparisons -- kept separate
# so the "standard" summary below stays comparable to other papers.
extra_corruptions = {"speckle_noise", "spatter", "gaussian_blur", "saturate"}


def evaluate_cifar10c(model, images, labels, batch_size=128):
    model.eval()
    correct = 0
    total = len(labels)
    with torch.no_grad():
        for start in range(0, total, batch_size):
            batch = torch.from_numpy(images[start:start + batch_size]).permute(0, 3, 1, 2).float().to(device) / 255.0
            batch = (batch - cifar_mean) / cifar_std
            target = torch.from_numpy(labels[start:start + batch_size]).long().to(device)
            correct += (model(batch).argmax(dim=1) == target).sum().item()
    return correct / total


# Note: each corruption .npy file stacks all 5 severity levels together
# (50,000 images), so the score below is one number averaged across
# severities, not a per-severity breakdown. That is fine for comparing w/
# vs w/o under identical conditions, but if you want numbers comparable to
# a published mCE table, split by severity (rows 0-9999 = severity 1,
# 10000-19999 = severity 2, and so on) and score each separately.

results_cifar10c = {"w": {}, "wo": {}}
model_w_eval = resnet18().to(device)
model_wo_eval = resnet18().to(device)

for net_idx in range(num_net):
    model_w_eval.load_state_dict(training_info_w[net_idx]["best_net"])
    model_wo_eval.load_state_dict(training_info_wo[net_idx]["best_net"])
    for path in corruption_paths:
        corruption = os.path.splitext(os.path.basename(path))[0]
        images = np.load(path)
        results_cifar10c["w"].setdefault(corruption, []).append(
            evaluate_cifar10c(model_w_eval, images, labels_c)
        )
        results_cifar10c["wo"].setdefault(corruption, []).append(
            evaluate_cifar10c(model_wo_eval, images, labels_c)
        )

for condition in ("w", "wo"):
    print(f"\n{condition} CIFAR-10-C accuracy")
    for corruption, scores in results_cifar10c[condition].items():
        if corruption in blur_corruptions:
            family = "blur"
        elif corruption in extra_corruptions:
            family = "extra"
        else:
            family = "other"
        print(f"{corruption:20s} [{family:5s}] {np.mean(scores):.4f} +/- {np.std(scores):.4f}")


In [ ]:
# Summary split: standard-15 average (excludes the 4 "extra" corruptions),
# further broken into blur vs non-blur so overlap with the training-time
# blur curriculum does not get conflated with genuine generalization.
for condition in ("w", "wo"):
    standard_scores = {c: np.mean(s) for c, s in results_cifar10c[condition].items() if c not in extra_corruptions}
    blur_scores = [v for c, v in standard_scores.items() if c in blur_corruptions]
    nonblur_scores = [v for c, v in standard_scores.items() if c not in blur_corruptions]
    print(f"{condition}: standard-15 avg = {np.mean(list(standard_scores.values())):.4f}, "
          f"blur-only avg = {np.mean(blur_scores):.4f}, non-blur avg = {np.mean(nonblur_scores):.4f}")


## Calibration & OOD detection benchmark (보너스: R의 원래 효과가 남아있는지 확인)

Random2 논문의 random-noise warm-up(R)이 원래 노리는 효과는 "확신도 보정(calibration)"이랑
"낯선 데이터 탐지(OOD detection)"였어. 지금 model_w는 R뿐 아니라 G, L까지 다 섞여있는데,
그래도 R의 원래 효과(ECE 개선, OOD 탐지력)가 남아있는지 이미 학습해둔 체크포인트로
재학습 없이 그냥 평가만 추가로 해보는 셀이야.

- **ECE (Expected Calibration Error)**: 낮을수록 좋음. "모델이 90% 확신했을 때 실제로 90% 정도
  맞히는가"를 재는 지표.
- **AUROC (SVHN을 OOD로 사용)**: CIFAR-10 test(정상 데이터)랑 SVHN(전혀 다른 데이터)을 넣었을 때,
  모델이 "이건 내가 아는 데이터", "이건 낯선 데이터"를 얼마나 잘 구분하는 확신도를 내는지 재는 지표.
  1에 가까울수록 좋음.

In [ ]:
from sklearn.metrics import roc_auc_score


def compute_ece(model, loader, n_bins=15):
    model.eval()
    confidences, predictions, labels_all = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs, labels = batch[0], batch[1]
            inputs, labels = inputs.to(device), labels.to(device)
            probs = torch.softmax(model(inputs), dim=1)
            conf, pred = probs.max(dim=1)
            confidences.append(conf.cpu().numpy())
            predictions.append(pred.cpu().numpy())
            labels_all.append(labels.cpu().numpy())
    confidences = np.concatenate(confidences)
    predictions = np.concatenate(predictions)
    labels_all = np.concatenate(labels_all)
    accuracies = (predictions == labels_all)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece


def get_max_confidence(model, loader):
    model.eval()
    confs = []
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0].to(device)
            probs = torch.softmax(model(inputs), dim=1)
            conf, _ = probs.max(dim=1)
            confs.append(conf.cpu().numpy())
    return np.concatenate(confs)


# SVHN: CIFAR-10이랑 같은 32x32 RGB라 별도 전처리 없이 같은 정규화(cifar mean/std)를 그대로 씀
svhn_dataset = torchvision.datasets.SVHN(
    root=dataset_dir, split="test", download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=True),
)
svhn_loader = DataLoader(svhn_dataset, batch_size=128, shuffle=False, num_workers=0)

ece_w, ece_wo = [], []
auroc_w, auroc_wo = [], []
model_eval = resnet18().to(device)

for net_idx in range(num_net):
    print(f"Computing calibration/OOD for network {net_idx+1}/{num_net}")

    model_eval.load_state_dict(training_info_w[net_idx]["best_net"])
    ece_w.append(compute_ece(model_eval, test_loader))
    conf_id = get_max_confidence(model_eval, test_loader)
    conf_ood = get_max_confidence(model_eval, svhn_loader)
    y_true = np.concatenate([np.ones_like(conf_id), np.zeros_like(conf_ood)])
    y_score = np.concatenate([conf_id, conf_ood])
    auroc_w.append(roc_auc_score(y_true, y_score))

    model_eval.load_state_dict(training_info_wo[net_idx]["best_net"])
    ece_wo.append(compute_ece(model_eval, test_loader))
    conf_id = get_max_confidence(model_eval, test_loader)
    conf_ood = get_max_confidence(model_eval, svhn_loader)
    y_true = np.concatenate([np.ones_like(conf_id), np.zeros_like(conf_ood)])
    y_score = np.concatenate([conf_id, conf_ood])
    auroc_wo.append(roc_auc_score(y_true, y_score))

print(f"\nECE   w/o degradation: {np.mean(ece_wo):.4f} +/- {np.std(ece_wo):.4f}")
print(f"ECE   w/ degradation : {np.mean(ece_w):.4f} +/- {np.std(ece_w):.4f}")
print(ttest_ind(ece_wo, ece_w))

print(f"\nAUROC (SVHN OOD) w/o degradation: {np.mean(auroc_wo):.4f} +/- {np.std(auroc_wo):.4f}")
print(f"AUROC (SVHN OOD) w/ degradation : {np.mean(auroc_w):.4f} +/- {np.std(auroc_w):.4f}")
print(ttest_ind(auroc_wo, auroc_w))


## Training curves

(The align/conflict panels from the bias-conflict notebook are dropped
here -- there is no bias split in plain CIFAR-10.)

In [ ]:
from custom.figure import plot_error

plt.figure(figsize=(60*mm, 30*mm))
data = [training_info_w[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0, 2.5)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.savefig(os.path.join(figure_dir, "train_loss.svg"))

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_w[net_idx]["train_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["train_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.ylim(0, 1)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")
plt.savefig(os.path.join(figure_dir, "train_acc.svg"))

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_w[net_idx]["test_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
data = [training_info_wo[net_idx]["test_acc"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
plt.axvline(x=epochs//10, linestyle='--', linewidth=0.5)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Clean Test Accuracy")
plt.savefig(os.path.join(figure_dir, "clean_test_acc.svg"))


## Feature-space check: does the model separate the 10 classes well?

Uses only the *best* checkpoint per network (per-epoch checkpoints are no
longer saved, so an epoch-by-epoch trajectory -- like the old silhouette
score cell -- is not possible anymore without saving per-epoch checkpoints
again). Also drops everything keyed on the bias attribute (decision rate,
bias-colored t-SNE, representation rate): plain CIFAR-10 has no such
attribute, so those numbers would not have meant anything.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA


def measure_feature_vector(model, loader, target_layers):
    model_copy = copy.deepcopy(model).to(device)
    features_dict = {layer: [] for layer in target_layers}
    hooks = []

    for name, module in model_copy.named_modules():
        if name in target_layers:
            def hook_fn(module, input, output, key=name):
                features_dict[key].append(torch.flatten(output, 1).detach().cpu().numpy())
            hooks.append(module.register_forward_hook(hook_fn))

    labels_list = []
    model_copy.eval()
    with torch.no_grad():
        for inputs, labels, _ in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _ = model_copy(inputs)
            labels_list.append(labels.cpu().numpy())

    for h in hooks:
        h.remove()

    for key in features_dict:
        features_dict[key] = np.concatenate(features_dict[key], axis=0)

    labels_array = np.concatenate(labels_list, axis=0)
    return features_dict, labels_array


def separability(X, y, cv=5, C=1.0, random_state=0):
    """Mean cross-validated accuracy of a linear SVM predicting y from X."""
    clf = make_pipeline(
        StandardScaler(),
        LinearSVC(C=C, max_iter=10_000, random_state=random_state)
    )
    scores = cross_val_score(clf, X, y, cv=cv)
    return scores.mean()


num_samples = 1000
rand_idx = np.random.choice(len(test_dataset), num_samples, replace=False)
small_test_dataset = torch.utils.data.Subset(test_dataset, rand_idx)
small_test_loader = DataLoader(small_test_dataset, batch_size=128, shuffle=False, num_workers=0)

intrinsic_sep_wo, intrinsic_sep_w = [], []

for net_idx in range(num_net):
    print(f"Measuring feature separability for Network {net_idx+1}/{num_net}")
    m_wo = resnet18().to(device)
    m_wo.load_state_dict(torch.load(os.path.join(save_dir, f"best_model_wo_{net_idx}.pth")))
    m_w = resnet18().to(device)
    m_w.load_state_dict(torch.load(os.path.join(save_dir, f"best_model_w_{net_idx}.pth")))

    feats_wo, labels_wo = measure_feature_vector(m_wo, small_test_loader, ["fc"])
    feats_w, labels_w = measure_feature_vector(m_w, small_test_loader, ["fc"])

    pca_wo = PCA(n_components=10).fit_transform(feats_wo["fc"])
    pca_w = PCA(n_components=10).fit_transform(feats_w["fc"])

    intrinsic_sep_wo.append(separability(pca_wo, labels_wo))
    intrinsic_sep_w.append(separability(pca_w, labels_w))

print(f"Intrinsic (class) separability w/o degradation: {np.mean(intrinsic_sep_wo):.4f} +/- {np.std(intrinsic_sep_wo):.4f}")
print(f"Intrinsic (class) separability w/ degradation:  {np.mean(intrinsic_sep_w):.4f} +/- {np.std(intrinsic_sep_w):.4f}")
print(ttest_ind(intrinsic_sep_wo, intrinsic_sep_w))
print(ranksums(intrinsic_sep_wo, intrinsic_sep_w))


In [ ]:
from sklearn.manifold import TSNE

net_idx = 0
model_wo_final = resnet18().to(device)
model_wo_final.load_state_dict(torch.load(os.path.join(save_dir, f"best_model_wo_{net_idx}.pth")))
model_w_final = resnet18().to(device)
model_w_final.load_state_dict(torch.load(os.path.join(save_dir, f"best_model_w_{net_idx}.pth")))

features_wo, labels_wo = measure_feature_vector(model_wo_final, test_loader, ["fc"])
features_w, labels_w = measure_feature_vector(model_w_final, test_loader, ["fc"])

tsne_wo = TSNE(n_components=2, random_state=1).fit_transform(features_wo["fc"])
tsne_w = TSNE(n_components=2, random_state=1).fit_transform(features_w["fc"])

random_order = np.random.permutation(len(tsne_wo))

plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_wo[random_order, 0], tsne_wo[random_order, 1], c=labels_wo[random_order], cmap='tab10', s=0.1)
plt.title("t-SNE (w/o degradation)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_wo.png"), dpi=300)

plt.figure(figsize=(30*mm, 30*mm))
plt.scatter(tsne_w[random_order, 0], tsne_w[random_order, 1], c=labels_w[random_order], cmap='tab10', s=0.1)
plt.title("t-SNE (w/ degradation)")
plt.xlabel("t-SNE axis 1")
plt.ylabel("t-SNE axis 2")
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.savefig(os.path.join(figure_dir, "tsne_w.png"), dpi=300)
